---
Task 2 — Meeting Transcript to Summary and Action Items 

Goal: Build a multi-agent pipeline that converts a raw meeting transcript into a concise meeting summary and structured action items. 

Steps: 
Create a Segmentation Agent to divide the transcript into logical sections based on speaker changes or discussion topics. 
Create a Summarizer Agent to generate a concise summary for each identified segment. 
Create an Action Extraction Agent to identify action items, owners, and due dates from the summarized content. 

Deliverables: 
One sample meeting transcript processed end-to-end. 
A concise 3–4 sentence meeting summary. 
A structured action item list including:  
Action Description 
Owner 
Due Date 
Status (optional) 

---

#### Setup


In [5]:
import os
import json
import time
from typing import TypedDict, List
import anthropic
from langgraph.graph import StateGraph, START, END
from getpass import getpass
from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("KEY")
base_url = os.getenv("BASE_URL")
model_name = os.getenv("MODEL")  # e.g., "claude-3-5-sonnet-20240620"

if not api_key or not model_name:
    raise ValueError("Missing 'KEY' or 'MODEL' environment variables! Please set them before running.")

# Initialize the Anthropic client (handles custom base URLs/proxies if provided)
if base_url:
    client = anthropic.Anthropic(api_key=api_key, base_url=base_url)
else:
    client = anthropic.Anthropic(api_key=api_key)

print(f"Anthropic client initialized successfully with model: {model_name}")

Anthropic client initialized successfully with model: global.anthropic.claude-haiku-4-5-20251001-v1:0


#### Define State Schema and Input Transcript

In [7]:
from typing import TypedDict, List

# Define the pipeline data state structure for Task 2
class MeetingPipelineState(TypedDict):
    raw_transcript: str
    segmented_topics: str
    overall_summary: str
    action_items: List[dict]

# Raw dialogue input block to analyze
raw_input_transcript = """
Sarah (Product Manager): Alright team, let's kick off our Q3 planning for the v4.3 analytics dashboard feature release. Our hard code freeze is set for August 15th, so we need to lock down engineering and testing timelines today.
David (Tech Lead): On the backend side, the core API migration to support large payload handling is about 80% complete. I still need to optimize the database query indexing and finalize the new asynchronous export endpoints. I should have the API documentation and endpoints completely finalized by July 20th.
Priya (QA Lead): Perfect. Once David hands off those API specifications on the 20th, my team will need to build the end-to-end automation test suites using our Pytest framework. We'll also run load tests on the async routes to prevent any 504 gateway timeouts under peak load. We should have the full test execution and bug verification wrapped up by August 5th.
Sarah (Product Manager): Sounds like a solid plan. I will compile these milestone dates and update our primary stakeholder progress dashboard by tomorrow morning, July 3rd, to keep leadership in the loop. Let's touch base again on Monday.
"""

print("State metadata tracks registered and text transcript successfully cached.")

State metadata tracks registered and text transcript successfully cached.


#### Implementing Node Worker

In [8]:
import json

def segmentation_node(state: MeetingPipelineState):
    """Uses Claude to segment raw streaming lines into logical topic sections."""
    response = client.messages.create(
        model=model_name,
        max_tokens=1500,
        temperature=0.1,
        system=(
            "You are an expert project manager. Parse the raw transcript and split it up "
            "chronologically into clearly labeled topic sections based on project updates or speaker focus."
        ),
        messages=[{"role": "user", "content": state["raw_transcript"]}]
    )
    print("\n[NODE TRACE: SEGMENTER] Chronological theme clusters structured.")
    return {"segmented_topics": response.content[0].text.strip()}

def summarizer_node(state: MeetingPipelineState):
    """Uses Claude to compress the segmented overview logs into a strict 3-4 sentence block."""
    prompt = f"Condense these conversation blocks into a high-level corporate summary:\n\n{state['segmented_topics']}"
    
    response = client.messages.create(
        model=model_name,
        max_tokens=512,
        temperature=0.2,
        system="You are an executive summary writing assistant. Write a summary that is exactly 3 to 4 sentences long total covering all milestones.",
        messages=[{"role": "user", "content": prompt}]
    )
    print("[NODE TRACE: SUMMARIZER] Executive summary brief finalized.")
    return {"overall_summary": response.content[0].text.strip()}

def action_extractor_node(state: MeetingPipelineState):
    """Uses Claude to construct an exact, strict JSON tracker matrix array."""
    prompt = f"Extract all action obligations from this context raw text:\n\n{state['raw_transcript']}"
    
    response = client.messages.create(
        model=model_name,
        max_tokens=1024,
        temperature=0.0,
        system=(
            "You are a meticulous task tracking data processor. Extract action items into a clean JSON array of objects. "
            "Do not output conversational wrapper sentences. Respond ONLY with raw JSON.\n"
            "Each object must match this exact key pattern: {\"description\": \"str\", \"owner\": \"str\", \"due_date\": \"str\", \"status\": \"str\"}"
        ),
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw_content = response.content[0].text.strip()
    
    # Text sanitization defense check to capture clean inner JSON boundaries
    if raw_content.startswith("```json"):
        raw_content = raw_content.split("```json")[1].split("```")[0].strip()
    elif raw_content.startswith("```"):
        raw_content = raw_content.split("```")[1].split("```")[0].strip()
        
    items = json.loads(raw_content)
    print("[NODE TRACE: EXTRACTOR] JSON milestone matrices extracted successfully.")
    return {"action_items": items}

#### Link State Architecture Graph

In [9]:
from langgraph.graph import StateGraph, START, END

# Initialize state graph layout context
task2_graph = StateGraph(MeetingPipelineState)

# Attach processing workers to graph address names
task2_graph.add_node("segmenter", segmentation_node)
task2_graph.add_node("summarizer", summarizer_node)
task2_graph.add_node("extractor", action_extractor_node)

# Map step execution path routes
task2_graph.add_edge(START, "segmenter")
task2_graph.add_edge("segmenter", "summarizer")
task2_graph.add_edge("summarizer", "extractor")
task2_graph.add_edge("extractor", END)

# Compile graph into a runnable component pipeline
meeting_pipeline = task2_graph.compile()
print("LangGraph meeting compilation pipeline locked and ready.")

LangGraph meeting compilation pipeline locked and ready.


#### Output

In [10]:
print("==========================================================")
print("             STARTING COMPREHENSIVE MEETING ANALYSIS       ")
print("==========================================================")

# Invoke processing execution across full pipeline
final_state = meeting_pipeline.invoke({"raw_transcript": raw_input_transcript})

print("\n" + "='"*30 + "\n")
print("📁 GENERATED SYSTEM DELIVERABLES\n")
print("### 📝 Executive Brief Summary")
print(final_state["overall_summary"])

print("\n### 🛠️ Extracted Task Matrix Plan")
print(f"{'Action Description':<70} | {'Owner':<23} | {'Due Date':<15} | {'Status':<12}")
print("-" * 128)

# Unpack individual action array dictionaries into console table columns
for item in final_state["action_items"]:
    desc = item.get("description", item.get("action_description", "N/A"))
    owner = item.get("owner", "N/A")
    date = item.get("due_date", "N/A")
    status = item.get("status", "N/A")
    print(f"{desc:<70} | {owner:<23} | {date:<15} | {status:<12}")

             STARTING COMPREHENSIVE MEETING ANALYSIS       

[NODE TRACE: SEGMENTER] Chronological theme clusters structured.
[NODE TRACE: SUMMARIZER] Executive summary brief finalized.
[NODE TRACE: EXTRACTOR] JSON milestone matrices extracted successfully.

='='='='='='='='='='='='='='='='='='='='='='='='='='='='='='

📁 GENERATED SYSTEM DELIVERABLES

### 📝 Executive Brief Summary
# Executive Summary: Q3 v4.3 Analytics Dashboard Feature Release

The v4.3 analytics dashboard feature release is progressing toward an August 15th code freeze, with backend development 80% complete on core API migration for large payload handling. QA testing is dependent on finalized API documentation and endpoints by July 20th, followed by comprehensive end-to-end automation and load testing to be completed by August 5th. Stakeholder communication and progress tracking will be updated by July 3rd to keep leadership informed of all critical milestones and dependencies throughout the release cycle.

### 🛠️ Ex